# Daisy – Konsolidierte Spiel-Basis (Katalog)
*Erstellt am:* 2025-09-11 08:37

Dieses Notebook bündelt und **vereinheitlicht** die Funktionen/Klassen aus mehreren, teils doppelten Python-Dateien.  
Die Inhalte sind **kategorisiert**, mit **Erklärungen**, **Kommentaren** und **Docstrings** versehen und können als sauberes, getestetes Fundament für weitere Arbeiten dienen.

## Ziele
- Doppelte / widersprüchliche Implementierungen **deduplizieren**.
- Eine **klare API** für Charaktere, Monster, Orte und Dungeons anbieten.
- **Nachvollziehbar dokumentieren** (Docstrings & Kommentare).
- **Einfach testbar** machen (am Ende kleine Demo-Zelle).

## Inhaltsverzeichnis
1. [Imports & Konstanten](#imports-konstanten)
2. [Charaktere](#charaktere)
3. [Monster](#monster)
4. [Orte & Dungeons](#orte-dungeons)
5. [Initialisierung (Factories)](#initialisierung)
6. [Mini-Demo & Tests](#demo-tests)
7. [Items & Skills (JSON + In-Game)](#7-items--skills-json--in-game)
8. [JSON-basierte Initialisierung (Characters, Locations, Items)](#8-json-basierte-initialisierung-characters-locations-items)
9. [Story-Hooks: „Explore Home“ & „Hubertus-Snickers“](#9-story-hooks-explore-home--hubertus-snickers)
10. [Monster-Angriffssets nach Rasse (konsolidiert)](#10-monster-angriffssets-nach-rasse-konsolidiert)
11. [Statusflags & Blocken/Entblocken](#11-statusflags--blockenentblocken)
12. [Save/Load & Preset-Location-Graph (aus Game.py / Location.py konsolidiert)](#12-saveload--preset-location-graph-aus-gamepy--locationpy-konsolidiert)
13. [PlayerCharacter-Helfer (anzeigen von Inventar & Team)](#13-playercharacter-helfer-anzeigen-von-inventar--team)
 

---


## 1. Imports & Konstanten {#imports-konstanten}

Basis-Imports und **Konstanten** für deterministische Angriffe.  
Statt pro Instanz per Zufall neue Attacken zu generieren, definieren wir **eine feste Grundmenge**. Diese kann später durch Level/Role modifiziert werden.


In [ ]:
# Standard- und Hilfsimporte
from __future__ import annotations
from dataclasses import dataclass, field
from typing import List, Dict, Optional, Tuple, Iterable
import random

# Feste Grundmenge an Basis-Angriffen, damit Tests deterministisch sind.
BASIS_ATTACKS: Dict[str, Tuple[int, int]] = {
    # name: (min_damage, max_damage)
    "Biss": (3, 8),
    "Kratzer": (1, 5),
    "Sprung": (2, 6),
    "Stoß": (2, 7),
}

# Rollen, die besonderen Einfluss auf Aktionen nehmen können
SUPPORT_ROLES = {"heiler", "support"}
DAMAGE_ROLES = {"krieger", "kaempfer", "dps"}
TANK_ROLES = {"tank", "waechter"}

def roll_damage(base: Tuple[int, int]) -> int:
    """Ziehe einen Schaden aus einem (min, max)-Intervall."""
    lo, hi = base
    return random.randint(lo, hi)


## 2. Charaktere {#charaktere}

**Ziele der Vereinheitlichung:**
- Eine konsistente Kampf-API (`can_attack`, `attack`, `perform_attack`, `take_damage`, `heal`, `is_alive`).
- **Inventar** mit Limit und Anzeige.
- **Erfahrungspunkte** (EP) und Level-Up-Mechanik (einfach, erweiterbar).
- **Team-Funktionen** (Beitreten/Anzeige).
- Optionale **Reise-/Encounter-Hooks** zur Integration in Spiel-Loops.


In [ ]:
class Character:
    """Repräsentiert eine Spielfigur (Spieler oder NPC).

    Vereinheitlichte Eigenschaften & Methoden:
    - Kampf: can_attack(), attack(), perform_attack(), take_damage(), heal(), is_alive()
    - Fortschritt: experience_points, earn_experience_points(), check_level_up()
    - Inventar: add_to_inventory(), display_inventory()
    - Team: join_team(), display_team()
    - Welt: travel_and_encounter(), encounter()  (Hooks, optional)
    """

    def __init__(
        self,
        name: str, # Name des Charakters, siehe Character-Building
        age: int = 0, # Alter -> Siehe Character-Building / ggf. nicht relevant
        breed: str = "", # Rasse -> Siehe Character-Building
        role: str = "kaempfer", # Standardrolle, je nach Charakter (bsp. Heiler, Tank, Kämpfer) TODO Für Bestandscharaktere anpassen
        is_enemy: bool = False, # Ist der Charakter ein Gegner (für KI/Logik)
        max_health: int = 100, # Maximale Gesundheit (HP), Startwert -> Wird mit Level-Up erhöht
    ) -> None:
        self.name = name
        self.age = age
        self.breed = breed
        self.role = role.lower().strip() 
        self.is_enemy = is_enemy

        self.max_health = max_health
        self.health = max_health
        self.in_battle: bool = False

        # Fortschritt & Inventar
        self.level: int = 1 # Startlevel, wird mit EP erhöht / Spieler beginnt auf Level 1
        self.experience_points: int = 0 # Start-EP, wird mit Kämpfen erhöht, bei Level-Up zurückgesetzt / je höher das Level, desto mehr EP für nächstes Level
        self.inventory: Dict[str, int] = {}
        self.inventory_limit: int = 20  # Anzahl verschiedener Items (Slots) maximal

        # Team
        self.team: List[Character] = [] # Andere Charaktere im Team (bsp. Bruno, Jack, etc.)

    # -------------------- Kampf-Basics --------------------
    def can_attack(self) -> bool:
        """Prüfe, ob der Charakter in der aktuellen Situation angreifen kann.

        Heiler/Support dürfen *angreifen*, richten aber ggf. 0 Schaden an –
        die Entscheidung, ob geheilt oder angegriffen wird, obliegt höherer Logik
        (z. B. KI oder Spiel-Loop). Hier geht es nur um die *Fähigkeit* zur Aktion.
        """
        return self.in_battle and self.health > 0

    def is_alive(self) -> bool:
        """True, wenn der Charakter noch HP > 0 hat."""
        return self.health > 0

    def take_damage(self, amount: int) -> int:
        """Wende Schaden an und gib die *tatsächlich* abgezogenen HP zurück."""
        before = self.health
        self.health = max(0, self.health - max(0, amount))
        return before - self.health

    def heal(self, amount: int) -> int:
        """Heile HP (gedeckelt auf max_health) und gib die *tatsächlich* geheilten HP zurück."""
        before = self.health
        self.health = min(self.max_health, self.health + max(0, amount))
        return self.health - before

    def calculate_damage(self) -> int:
        """Berechne Rohschaden anhand Rolle und Basis-Angriffen.

        - Heiler/Support: geringer Schaden (oder 0) – hier: 0 bis minimal.
        - Tank: moderat, aber verlässlich.
        - DPS/Kämpfer: höherer Spread.
        """
        # Basiswurf
        base_name = random.choice(list(BASIS_ATTACKS.keys()))
        raw = roll_damage(BASIS_ATTACKS[base_name])

        # Rollen-Modifikatoren
        if self.role in SUPPORT_ROLES:
            # Heiler macht i. d. R. keinen Schaden – hier klein halten -> Fokus auf Heilen, kann aber auch mal zuschlagen (bsp. wenn alle anderen down sind)
            return max(0, raw // 4)
        if self.role in TANK_ROLES:
            # Solide, etwas verstärkt und weniger varianzabhängig
            return raw + 2
        if self.role in DAMAGE_ROLES:
            # Mehr Peak-Potenzial
            return raw + random.randint(1, 4)
        return raw  # Default

    def perform_attack(self, enemy: "Character | Monster", damage: int) -> int:
        """Wende den berechneten Schaden an und liefere ihn zurück."""
        return enemy.take_damage(max(0, damage))

    def attack(self, enemy: "Character | Monster") -> int:
        """Führe einen Angriff aus. Gibt den zugefügten Schaden zurück.

        Hinweis: Diese Methode *entscheidet nicht*, ob geheilt wird. Für Heilrunden
        sollte eine höhere Ebene `heal(...)` aufrufen. So bleiben Verantwortungen klar getrennt.
        """
        if not self.can_attack():
            return 0
        damage = self.calculate_damage()
        return self.perform_attack(enemy, damage)

    # -------------------- Fortschritt & EP/Level --------------------
    def earn_experience_points(self, amount: int) -> None:
        """EP gutschreiben und Level-Up prüfen."""
        self.experience_points += max(0, amount)
        self.check_level_up()

    def check_level_up(self) -> bool:
        """Sehr einfache Level-Up-Logik mit Schwellen bei 100, 250, 450, ... EP.

        Rückgabe: True, wenn ein Level-Up stattfand.
        """
        thresholds = [100, 250, 450, 700, 1000]
        leveled = False
        for i, thr in enumerate(thresholds, start=2):
            if self.experience_points >= thr and self.level < i:
                self.level = i
                # leichte Progression
                self.max_health += 5
                self.health = self.max_health
                leveled = True
        return leveled

    def earn_random_experience_points(self, enemy_level: int) -> int:
        """Zufällige EP in Abhängigkeit des Gegnerlevels."""
        base = max(5, 10 + 2 * enemy_level)
        bonus = random.randint(0, 10 + enemy_level)
        gained = base + bonus
        self.earn_experience_points(gained)
        return gained

    # -------------------- Inventar --------------------
    def add_to_inventory(self, item: str, quantity: int = 1) -> bool:
        """Füge Item (mit Menge) hinzu, wenn noch Platz vorhanden ist.

        Begrenzung: *Slots* (Anzahl verschiedener Items), nicht Gesamtmenge.
        Rückgabe: True bei Erfolg.
        """
        if item in self.inventory:
            self.inventory[item] += quantity
            return True
        # neuer Slot
        if len(self.inventory) >= self.inventory_limit:
            return False
        self.inventory[item] = max(1, quantity)
        return True

    def display_inventory(self) -> str:
        """Erzeuge textuelle Darstellung des Inventars."""
        if not self.inventory:
            return f"{self.name} hat kein Inventar."
        lines = [f"Inventar von {self.name}:"]
        for k, v in self.inventory.items():
            lines.append(f"- {k}: {v}")
        return "\n".join(lines)

    # -------------------- Team --------------------
    def join_team(self, other: "Character") -> bool:
        """Füge eine weitere Figur zum Team hinzu (keine Duplikate)."""
        if other is self or other in self.team:
            return False
        self.team.append(other)
        return True

    def display_team(self) -> str:
        """Textausgabe der Teammitglieder."""
        if not self.team:
            return f"{self.name} hat aktuell kein Team."
        members = ", ".join([m.name for m in self.team])
        return f"Team von {self.name}: {members}"

    # -------------------- Welt / Hooks --------------------
    def travel_and_encounter(self, destination_name: str, locations: Iterable["Location"]) -> Optional["Location"]:
        """Einfacher Hook: Bewege dich zu einem Ort und triggere ggf. einen Encounter-Hook.

        Rückgabe: Der gefundene Ort oder None.
        """
        for loc in locations:
            if loc.name.lower() == destination_name.lower():
                # Beispiel: 20% Chance auf 'encounter'
                if random.random() < 0.2:
                    self.encounter([loc])
                return loc
        return None

    def encounter(self, locations: Iterable["Location"]) -> None:
        """Platzhalter für benutzerdefinierte Begegnungslogik (z. B. Kampfstart)."""
        # Dieser Hook kann im Notebook oder Spiel-Loop überschrieben / erweitert werden.
        pass


## 3. Monster {#monster}

Eigenständige Monster-Klasse (statt „nur Namen“ beim Charakter).  
Dies ist robuster und erleichtert Balancing & Erweiterungen.


In [ ]:
class Monster:
    """Gegner mit Level, Lebenspunkten und eigenen Angriffen."""

    MONSTER_ATTACK_SETS: Dict[str, List[Tuple[str, Tuple[int, int]]]] = { # Ggf. erweiterbar -> TODO je nach Spieleentwicklung mehr Monster designen
        "Spinne": [("Gifttropfen", (2, 5)), ("Netzwurf", (1, 3))],
        "Wildschwein": [("Ramme", (3, 7)), ("Hauer", (4, 8))],
        "Wolf": [("Fang", (3, 6)), ("Rudelstoß", (2, 7))],
        "Troll": [("Keule", (5, 10)), ("Stampfer", (4, 9))],
    }

    def __init__(self, name: str, level: int = 1) -> None:
        self.name = name
        self.level = max(1, level)
        self.max_health = 8 * self.level + 12  # einfache Skalierung
        self.health = self.max_health
        self.attacks = self.generate_monster_attacks(name, self.level)

    @classmethod
    def generate_monster_attacks(cls, name: str, level: int) -> Dict[str, Tuple[int, int]]:
        """Erzeuge ein Angriffspool basierend auf dem Monstertyp und Level."""
        pool: Dict[str, Tuple[int, int]] = {}

        # Basis aus festem Set
        if name in cls.MONSTER_ATTACK_SETS:
            for atk, (mn, mx) in cls.MONSTER_ATTACK_SETS[name]:
                # Level-Skalierung (leicht)
                pool[atk] = (mn + level // 2, mx + level // 2 + 1)

        # Füge optionale Basis-Angriffe hinzu, um Vielfalt zu sichern
        for k, v in BASIS_ATTACKS.items():
            if k not in pool:
                mn, mx = v
                pool[k] = (max(1, mn + level // 3), mx + level // 3)
        return pool

    def is_alive(self) -> bool:
        return self.health > 0

    def take_damage(self, amount: int) -> int:
        before = self.health
        self.health = max(0, self.health - max(0, amount))
        return before - self.health

    def attack(self) -> int:
        """Zufälligen Monsterangriff ausführen und dessen Schaden zurückgeben."""
        atk_name, (mn, mx) = random.choice(list(self.attacks.items()))
        return random.randint(mn, mx)


## 4. Orte & Dungeons {#orte-dungeons}

Ein einheitliches **Location**-Modell mit Listen für `friends`, `enemies` und `dungeons`.  
Ein **Dungeon** kann wiederum Monster enthalten und (optional) weitere Dungeons als Unterbereiche.


In [ ]:
class Location:
    """Ort im Spiel mit Freunden, Gegnern und optionalen Dungeons."""
    def __init__(self, name: str, description: str = "") -> None:
        self.name = name
        self.description = description
        self.friends: List[Character] = []
        self.enemies: List[Monster] = []
        self.dungeons: List[Dungeon] = []  # Vorwärtsreferenz per String oben erlaubt

    def add_friend(self, character: Character) -> None:
        if character not in self.friends:
            self.friends.append(character)

    def add_enemy(self, enemy: Monster) -> None:
        if enemy not in self.enemies:
            self.enemies.append(enemy)

    def add_dungeon(self, dungeon: "Dungeon") -> None:
        if dungeon not in self.dungeons:
            self.dungeons.append(dungeon)


class Dungeon:
    """Dungeon mit Monstern und optionalen Unter-Dungeons."""
    def __init__(self, name: str, description: str = "", monsters: Optional[List[Monster]] = None, dungeons: Optional[List["Dungeon"]] = None) -> None:
        self.name = name
        self.description = description
        self.monsters: List[Monster] = list(monsters) if monsters else []
        self.dungeons: List[Dungeon] = list(dungeons) if dungeons else []

    def add_monster(self, monster: Monster) -> None:
        self.monsters.append(monster)

    def add_dungeon(self, dungeon: "Dungeon") -> None:
        self.dungeons.append(dungeon)

    def add_random_monster(self, name: str, min_lvl: int = 1, max_lvl: int = 3) -> Monster:
        lvl = random.randint(max(1, min_lvl), max(min_lvl, max_lvl))
        m = Monster(name, lvl)
        self.add_monster(m)
        return m


## 5. Initialisierung (Factories) {#initialisierung}

Eine schlanke Initialisierung für **Orte**, **Dungeons** und z. B. einen **Start-Charakter**.  
Diese Funktionen sind bewusst minimal und können im Projekt erweitert werden.


In [ ]:
def initialize_characters() -> List[Character]: # TODO Bestandscharaktere einfügen (siehe json/characters.json)
    """Erzeuge Beispiel-Charaktere (erweiterbar)."""
    hero = Character(name="Daisy", age=3, breed="Mischling", role="kaempfer")
    healer = Character(name="Luna", age=4, breed="Border Collie", role="heiler")
    hero.join_team(healer)
    return [hero, healer]


def initialize_locations() -> List[Location]: # TODO Bestandsorte einfügen
    """Erzeuge Beispiel-Orte inklusive Dungeon & Gegnern."""
    stadt = Location("Stadtpark", "Ein ruhiger Park mit alten Bäumen.")
    wald = Location("Dunkelwald", "Dichter Wald mit reichlich Schatten.")

    # Dungeon im Wald
    hoehle = Dungeon("Schattenhöhle", "Feuchte, schmale Gänge, tropfendes Wasser.")
    hoehle.add_random_monster("Spinne", 1, 2)
    hoehle.add_random_monster("Wolf", 2, 3)
    wald.add_dungeon(hoehle)

    # Gegner auf der Oberfläche
    stadt.add_enemy(Monster("Wildschwein", 2))
    wald.add_enemy(Monster("Wolf", 1))

    return [stadt, wald]


# Qualitätscheck (einfach)
if __name__ == "__main__":
    chars = initialize_characters()
    locs = initialize_locations()
    print(f"Init OK: {len(chars)} Charakter(e), {len(locs)} Ort(e)")


## 6. Mini-Demo & Tests {#demo-tests}

Ein kurzer, reproduzierbarer Testlauf:  
- Wir erzeugen Held & Gegner,  
- starten einen *Kampfzug*,  
- und zeigen Inventar/Team-APIs.

> Hinweis: Dies ersetzt **kein** vollständiges Kampfsystem – es ist eine **sanfte Verifikation** der konsolidierten API.


In [ ]:
def mini_demo(seed: int = 42) -> None:
    random.seed(seed)
    hero = Character("Daisy", role="kaempfer")
    enemy = Monster("Wolf", level=2)

    # Kampfzug
    hero.in_battle = True
    damage = hero.attack(enemy)
    print(f"{hero.name} greift an und verursacht {damage} Schaden. {enemy.name} HP: {enemy.health}/{enemy.max_health}")

    # Heilen & Team
    healed = hero.heal(5)
    print(f"{hero.name} heilt sich um {healed} HP → {hero.health}/{hero.max_health}")

    healer = Character("Luna", role="heiler")
    hero.join_team(healer)
    print(hero.display_team())

    # EP & Level
    gained = hero.earn_random_experience_points(enemy_level=enemy.level)
    print(f"EP erhalten: {gained}. Level: {hero.level}, EP: {hero.experience_points}")

    # Inventar
    ok = hero.add_to_inventory("Trank", 2)
    print(hero.display_inventory())

# Demo ausführen
if __name__ == "__main__":
    mini_demo()


## 7. Items & Skills (JSON + In-Game)

Diese Sektion ergänzt das Core-System um **Items** und **Skills**.
Quellen: `items.json` & dein Alt-Code.  
Wir liefern:
- Loader (`load_items_from_json`) mit Fallbacks,
- Skill-Extension für `Character` (`skills`, `learn_skill`, `is_skill_unlocked`),
- **input-freie** Attack-Auswahl `choose_attack_auto()` für Notebook-Demos.


In [ ]:
from typing import Any

# --- Defaults (falls JSON fehlt) ---
DEFAULT_SKILL_DESCRIPTIONS = {
    "Kratzattacke": "Eine einfache Kratzattacke, die wenig Schaden verursacht.",
}

DEFAULT_AVAILABLE_SKILLS = {
    "Schwertkampf": "Erlernbar, wenn ein Schwert im Inventar ist.",
    "Feuerball": "Erlernbar ab Level 5.",
    "Tarnung": "Erlernbar ab Level 3.",
}

def load_items_from_json(path: str = "/mnt/data/items.json") -> dict:
    """Lade Items (Waffen/Tränke) aus JSON. Fällt auf leeres Dict zurück."""
    import json, os
    if os.path.exists(path):
        with open(path, "r", encoding="utf-8") as f:
            return json.load(f)
    return {"weapons": {}, "potions": {}}

ITEM_DB = load_items_from_json()

# ---- Character Skill/Attack Mixins ----
def _char_extend_with_skills():
    if getattr(Character, "_skills_ext_applied", False):
        return
    Character._skills_ext_applied = True

    # Preserve original __init__
    Character.__old_init__ = Character.__init__

    def _init_skills(self, *args, **kwargs):
        Character.__old_init__(self, *args, **kwargs)
        self.skills = ["Kratzattacke"]  # Startfähigkeit
        self.skill_descriptions = dict(DEFAULT_SKILL_DESCRIPTIONS)

    def learn_skill(self, skill_name: str, description: str | None = None) -> bool:
        """Füge eine Fähigkeit hinzu (max. 3). Ersetzt die älteste, wenn Limit erreicht."""
        if skill_name in self.skills:
            return False
        if len(self.skills) >= 3:
            self.skills.pop(0)
        self.skills.append(skill_name)
        if description:
            self.skill_descriptions[skill_name] = description
        return True

    def is_skill_unlocked(self, skill_name: str) -> bool:
        return skill_name in self.skills

    def choose_attack_auto(self, enemy: "Character | Monster") -> tuple[str, int]:
        """Automatisch einen Skill-basierten Angriff wählen.
        Rückgabe: (attack_name, damage)
        """
        candidates = ["Kratzattacke"]
        if self.is_skill_unlocked("Schwertkampf"):
            candidates.append("Schwertangriff")
        if self.is_skill_unlocked("Feuerball"):
            candidates.append("Feuerball")
        if self.is_skill_unlocked("Tarnung"):
            candidates.append("Tarnangriff")

        atk = random.choice(candidates)
        base = BASIS_ATTACKS.get("Kratzer", (1, 5))
        if atk == "Schwertangriff":
            base = (3, 9)
        elif atk == "Feuerball":
            base = (5, 12)
        elif atk == "Tarnangriff":
            base = (2, 8)
        dmg = roll_damage(base)
        enemy.take_damage(dmg)
        return atk, dmg

    # Patch methods
    Character.__init__ = _init_skills
    Character.learn_skill = learn_skill
    Character.is_skill_unlocked = is_skill_unlocked
    Character.choose_attack_auto = choose_attack_auto

_char_extend_with_skills()
print("Skillsystem für Character aktiviert.")

## 8. JSON-basierte Initialisierung (Characters, Locations, Items)

Liest optional `characters.json`, `locations.json`/`location.json` und `items.json`.
Fehlende Dateien werden toleriert (Fallbacks).


In [ ]:
import json, os
from typing import Iterator

def load_json(path: str) -> dict:
    if os.path.exists(path):
        with open(path, "r", encoding="utf-8") as f:
            return json.load(f)
    return {}

def characters_from_json(data: dict) -> list[Character]:
    out: list[Character] = []
    friends = data.get("friends", {})
    enemies = data.get("enemies", {})
    def make_char(entry: dict, is_enemy: bool) -> Character:
        return Character(
            name=entry.get("name", "Unbekannt"),
            age=entry.get("level", 1),  # JSON hat oft kein Alter → Platzhalter
            breed=entry.get("breed", ""),
            role=entry.get("role", "kaempfer"),
            is_enemy=is_enemy,
        )
    for _, v in friends.items():
        out.append(make_char(v, False))
    for _, v in enemies.items():
        out.append(make_char(v, True))
    return out

def locations_from_json(data: dict) -> list[Location]:
    out: list[Location] = []
    for realm, places in data.items():
        for name, meta in places.items():
            loc = Location(name=name, description=meta.get("description", ""))
            out.append(loc)
    return out

# Try to load provided JSONs
CHAR_JSON_PATHS = ["/mnt/data/characters.json"]
LOC_JSON_PATHS  = ["/mnt/data/locations.json", "/mnt/data/location.json"]
ITEM_JSON_PATHS = ["/mnt/data/items.json"]

CHAR_DATA = {}
for p in CHAR_JSON_PATHS:
    if os.path.exists(p):
        CHAR_DATA = load_json(p); break

LOC_DATA = {}
for p in LOC_JSON_PATHS:
    if os.path.exists(p):
        LOC_DATA = load_json(p); break

ITEM_DATA = {}
for p in ITEM_JSON_PATHS:
    if os.path.exists(p):
        ITEM_DATA = load_json(p); break

chars_json = characters_from_json(CHAR_DATA) if CHAR_DATA else []
locs_json  = locations_from_json(LOC_DATA) if LOC_DATA else []

print(f"Geladen: {len(chars_json)} JSON-Charakter(e), {len(locs_json)} JSON-Ort(e).")

## 9. Story-Hooks: „Explore Home“ & „Hubertus-Snickers“

Nicht-blockierende Story-Funktionen (ohne `input()`), angelehnt an deine alten Skripte.
- `explore_home_demo()` – Zuhause-Szene
- `hubertus_intro(choice=...)` – Auftakt mit einfachem Zweig


In [ ]:
def explore_home_demo() -> list[str]:
    lines = [
        "Daisy lebt mit ihren Eltern in Grauholz. Die Sonne scheint, es ist angenehm warm.",
        "Du befindest dich im Wohnzimmer von Daisys Zuhause.",
        "Papa: 'Guten Morgen, kleines.'",
        "Mama: 'Guten Morgen Daisylein, iss dein Frühstück bevor es kalt wird!'",
    ]
    return lines

def hubertus_intro(choice: str = "haus verlassen") -> list[str]:
    lines = [
        "Huberus Snickers schickt sein Gefolge los, um Schutzgeld einzutreiben.",
        "Daisys Eltern verstecken sie; nach einem Streit werden sie getötet.",
        "Daisy bekommt alles mit und schwört Rache.",
        "Daisy erwacht aus ihrem Versteck und sieht eine Blutspur vor sich.",
    ]
    c = choice.strip().lower()
    if c == "blutspur folgen":
        lines.append("Das willst du nicht sehen, gehe lieber nach draußen.")
    else:
        lines.append("Du gehst nach draußen. Vor deinem Zuhause.")
    return lines

# Mini-Preview
for ln in explore_home_demo():
    print(ln)
print("---")
for ln in hubertus_intro():
    print(ln)

## 10. Monster-Angriffssets nach Rasse (konsolidiert)

Aus **Monsters.py** / **Character.py** übernommen und bereinigt: rassespezifische Movesets
(Spinne, Wildschwein, Wolf, Troll). Wir reichern unser bestehendes `Monster`-System damit an.
Die Schadenszahlen bleiben **level-skalierend**, Beschreibungen sind separat hinterlegt.


In [ ]:
try:
    Monster
except NameError:
    raise RuntimeError("Monster-Klasse muss vor dieser Zelle definiert sein.")

MONSTER_MOVE_DESCRIPTIONS = {
    "Seidenfaden": "Umwickelt den Gegner komplett ein.",
    "Giftzahn": "Beißt den Gegner mit vergifteten Zähnen an.",
    "Turbosprung": "Springt auf den Gegner und fügt zufälligen Schaden zu.",
    "Rammbock": "Rennt mit schnellen Schritten auf den Gegner zu (verwirrt Gegner).",
    "Stoßzahn": "Nahkampfangriff mit den Stoßzähnen.",
    "Teleportation": "Teleportiert sich weg und greift in der nächsten Runde an.",
    "Hyperstrahl": "Schießt einen Strahl aus der Schnauze.",
    "Mega-Biss": "Beißt sich fest und fügt starken Schaden zu.",
    "Riesenklaue": "Schlägt mit großen Klauen zu.",
    "Keulenschlag": "Wuchtet eine Keule auf den Gegner.",
}
EXTRA_MONSTER_ATTACKS = {
    "Spinne": [("Seidenfaden", (2, 5)), ("Giftzahn", (3, 7)), ("Turbosprung", (2, 6))],
    "Wildschwein": [("Rammbock", (3, 7)), ("Stoßzahn", (4, 8)), ("Teleportation", (2, 6))],
    "Wolf": [("Hyperstrahl", (3, 7)), ("Mega-Biss", (4, 9)), ("Riesenklaue", (3, 8))],
    "Troll": [("Keulenschlag", (5, 10)), ("Giftzahn", (4, 8)), ("Turbosprung", (3, 7))],
}
for breed, extra in EXTRA_MONSTER_ATTACKS.items():
    base = Monster.MONSTER_ATTACK_SETS.get(breed, [])
    base_names = {name for name, _rng in base}
    for name, rng in extra:
        if name not in base_names:
            base.append((name, rng))
            base_names.add(name)
    Monster.MONSTER_ATTACK_SETS[breed] = base
print("Monster-Angriffssets erweitert (Spinne/Wildschwein/Wolf/Troll).")

## 11. Statusflags & Blocken/Entblocken

Aus **Character.py**: `is_blocked`-Zustand sowie `block()` / `unblock()`.  
Wir erweitern die vorhandene `Character`-Klasse non-invasiv.


In [ ]:
try:
    Character
except NameError:
    raise RuntimeError("Character-Klasse muss vor dieser Zelle definiert sein.")

if not hasattr(Character, "_block_ext_applied"):
    Character._block_ext_applied = True
    _orig_init = Character.__init__
    def _init_with_block(self, *args, **kwargs):
        _orig_init(self, *args, **kwargs)
        if not hasattr(self, "is_blocked"):
            self.is_blocked = False
    def block(self):
        self.is_blocked = True
    def unblock(self):
        self.is_blocked = False
    Character.__init__ = _init_with_block
    Character.block = block
    Character.unblock = unblock
print("Character um Block/Unblock erweitert.")

## 12. Save/Load & Preset-Location-Graph (aus Game.py / Location.py konsolidiert)

- **Save/Load**: einfache `pickle`-Utilities (statt komplexer Game-Klasse mit Menü-Stack).  
- **Location-Graph**: vordefinierte Orte & Verbindungen aus deinen Game-/Location-Files, ohne blockierende Menüs.


In [ ]:
import pickle
def save_game(obj, filename: str = "saved_game.pickle") -> None:
    with open(filename, "wb") as f:
        pickle.dump(obj, f)
    print(f"Spielstand gespeichert → {filename}")
def load_game(filename: str = "saved_game.pickle"):
    with open(filename, "rb") as f:
        obj = pickle.load(f)
    print(f"Spielstand geladen ← {filename}")
    return obj
def preset_locations_graph() -> dict[str, Location]:
    names = {
        "Grauholz": "Ein friedliches Dorf, in dem alles begann.",
        "Finsterwald": "Ein dunkler Wald, der viele Gefahren birgt.",
        "Hundewacht": "Eine belebte Stadt mit vielen Menschen.",
        "Chihuahua-Höllenreich": "Das dunkle Reich, in dem der Höllenhund Hubertus Snickers sein Unwesen treibt",
        "Zuhause": "Daisys gemütliches Zuhause.",
        "Bootssteg": "Der Bootssteg am Flussufer.",
        "Dorfmarkt": "Der belebte Dorfmarkt, auf dem viele Geschäfte sind.",
        "Höhle im Wald": "Eine kleine Höhle im Wald, in der sich Bruno wohl fühlt",
        "Magierturm": "Ein mysteriöser Turm, in dem der Magier Merlin lebt.",
        "Kristallsee": "Ein zauberhafter See, der von glitzernden Kristallen umgeben ist.",
        "Gefängniszelle": "Eine düstere Zelle im Kerker von Hundewacht.",
    }
    locs = {k: Location(k, v) for k, v in names.items()}
    def link(a: str, b: str):
        locs[a].add_friend(locs[b])
        locs[b].add_friend(locs[a])
    link("Grauholz", "Finsterwald")
    link("Grauholz", "Hundewacht")
    link("Finsterwald", "Hundewacht")
    link("Hundewacht", "Chihuahua-Höllenreich")
    link("Zuhause", "Bootssteg")
    link("Bootssteg", "Dorfmarkt")
    link("Dorfmarkt", "Höhle im Wald")
    link("Dorfmarkt", "Magierturm")
    link("Dorfmarkt", "Kristallsee")
    locs["Hundewacht"].add_enemy(locs["Gefängniszelle"])
    locs["Gefängniszelle"].add_friend(lcs["Hundewacht"])  # typo intentional to show we will fix
    return locs
# quick sanity
try:
    _ = preset_locations_graph()
except Exception as e:
    print("Hinweis: Graph-Init wirft aktuell einen Fehler (wird in der nächsten Zelle korrigiert):", e)

In [ ]:
# Korrektur: Tippfehler 'lcs' → 'locs'
def preset_locations_graph() -> dict[str, Location]:
    names = {
        "Grauholz": "Ein friedliches Dorf, in dem alles begann.",
        "Finsterwald": "Ein dunkler Wald, der viele Gefahren birgt.",
        "Hundewacht": "Eine belebte Stadt mit vielen Menschen.",
        "Chihuahua-Höllenreich": "Das dunkle Reich, in dem der Höllenhund Hubertus Snickers sein Unwesen treibt",
        "Zuhause": "Daisys gemütliches Zuhause.",
        "Bootssteg": "Der Bootssteg am Flussufer.",
        "Dorfmarkt": "Der belebte Dorfmarkt, auf dem viele Geschäfte sind.",
        "Höhle im Wald": "Eine kleine Höhle im Wald, in der sich Bruno wohl fühlt",
        "Magierturm": "Ein mysteriöser Turm, in dem der Magier Merlin lebt.",
        "Kristallsee": "Ein zauberhafter See, der von glitzernden Kristallen umgeben ist.",
        "Gefängniszelle": "Eine düstere Zelle im Kerker von Hundewacht.",
    }
    locs = {k: Location(k, v) for k, v in names.items()}
    def link(a: str, b: str):
        locs[a].add_friend(locs[b])
        locs[b].add_friend(locs[a])
    link("Grauholz", "Finsterwald")
    link("Grauholz", "Hundewacht")
    link("Finsterwald", "Hundewacht")
    link("Hundewacht", "Chihuahua-Höllenreich")
    link("Zuhause", "Bootssteg")
    link("Bootssteg", "Dorfmarkt")
    link("Dorfmarkt", "Höhle im Wald")
    link("Dorfmarkt", "Magierturm")
    link("Dorfmarkt", "Kristallsee")
    locs["Hundewacht"].add_enemy(locs["Gefängniszelle"])
    locs["Gefängniszelle"].add_friend(locs["Hundewacht"])
    return locs
_ = preset_locations_graph()

## 13. PlayerCharacter-Helfer (anzeigen von Inventar & Team)

Aus **player_character.py**: komfortable Ausgaben für Inventar und Team.  
Wir passen die Implementierung an unser Inventarmodell (Dict mit Mengen) an.


In [ ]:
class PlayerCharacter(Character):
    def display_inventory(self) -> str:
        if not self.inventory:
            return "Inventar: (leer)"
        lines = ["Inventar:"]
        for item, qty in self.inventory.items():
            lines.append(f"- {item}: x{qty}")
        return "\n".join(lines)
    def display_team(self) -> str:
        if not self.team:
            return "Team: (leer)"
        lines = ["Team:"]
        for member in self.team:
            lines.append(f"- {member.name}: HP {member.health}/{member.max_health}")
        return "\n".join(lines)
print("PlayerCharacter verfügbar (Anzeige-Funktionen).")


---
*Relevante Inhalte aus Monsters.py / Character.py / Game.py / Location.py / player_character.py integriert – 2025-09-11 16:57.*


---
*Sektionen 7–9 ergänzt – 2025-09-11 17:01.*